In [ ]:
import json
import glob
import random
from tqdm import tqdm

DATA_DIR = "data/*.json"        # folder with 1000 match files
OUT_JSONL = "cricket_instructions.jsonl"

# -------------------------
# Commentary generator
# -------------------------
def delivery_to_commentary(d):
    batter = d["batter"]
    bowler = d["bowler"]
    runs = d["runs"]["total"]

    text = f"{bowler} to {batter}, "

    if "wickets" in d:
        w = d["wickets"][0]
        kind = w["kind"]
        fielder = w.get("fielders", [{}])[0].get("name", "")
        if fielder:
            return text + f"OUT! {batter} is {kind}, caught by {fielder}."
        return text + f"OUT! {batter} is {kind}."

    if runs == 0:
        return text + "no run."
    if runs == 4:
        return text + "FOUR!"
    if runs == 6:
        return text + "SIX!"
    return text + f"{runs} run{'s' if runs > 1 else ''}."


# -------------------------
# Instruction pool
# -------------------------
INSTRUCTIONS = [
    "Describe the cricket delivery in natural language.",
    "Write ball-by-ball commentary for this delivery.",
    "Summarize what happened on this ball.",
    "Explain the outcome of this cricket delivery.",
    "Generate live commentary for this ball."
]


rows = []
files = glob.glob(DATA_DIR)

print(f"📂 Found {len(files)} match files")

for file in tqdm(files, desc="Processing matches"):
    with open(file, "r", encoding="utf-8") as f:
        match = json.load(f)

    for inning in match["innings"]:
        team = inning["team"]

        for over in inning["overs"]:
            over_no = over["over"]

            for ball_no, d in enumerate(over["deliveries"], start=1):

                input_payload = {
                    "match_file": file,
                    "team": team,
                    "over": over_no,
                    "ball": ball_no,
                    "batter": d["batter"],
                    "bowler": d["bowler"],
                    "runs": d["runs"],
                    "extras": d.get("extras", {}),
                    "wickets": d.get("wickets", [])
                }

                rows.append({
                    "instruction": random.choice(INSTRUCTIONS),
                    "input": json.dumps(input_payload, ensure_ascii=False),
                    "output": delivery_to_commentary(d)
                })


# -------------------------
# Write JSONL
# -------------------------
with open(OUT_JSONL, "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r) + "\n")

print(f"✅ Created {len(rows)} instruction samples")
print(f"📄 Saved to {OUT_JSONL}")


In [ ]:
folder_path = "/mnt/c/Users/pragn/Downloads/Gemma_finetune"
print(os.listdir(folder_path))

In [ ]:
import json
import os
import random
from pathlib import Path

DATA_DIR = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/data"         # folder with 1000+ match JSON files
OUTPUT_FILE = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/train.jsonl"

INSTRUCTIONS = [
    "Explain the outcome of this cricket delivery.",
    "Write ball-by-ball commentary for this delivery.",
    "Generate live commentary for this ball.",
    "Describe the cricket delivery in natural language.",
    "Summarize what happened on this ball."
]

def describe_delivery(delivery):
    batter = delivery["batter"]
    bowler = delivery["bowler"]
    runs = delivery["runs"]["batter"]
    total = delivery["runs"]["total"]
    extras = delivery.get("extras", {})
    wickets = delivery.get("wickets", [])

    # WICKET
    if wickets:
        w = wickets[0]
        kind = w["kind"]
        if "fielders" in w:
            fielder = w["fielders"][0]["name"]
            return f"{bowler} to {batter}, OUT! {kind}, taken by {fielder}."
        else:
            return f"{bowler} to {batter}, OUT! {kind}."

    # EXTRAS
    if extras:
        if "wides" in extras:
            return f"{bowler} to {batter}, {extras['wides']} wides."
        if "legbyes" in extras:
            return f"{bowler} to {batter}, {extras['legbyes']} leg byes."
        if "byes" in extras:
            return f"{bowler} to {batter}, {extras['byes']} byes."

    # NORMAL RUNS
    if runs == 0:
        return f"{bowler} to {batter}, no run."
    if runs == 4:
        return f"{bowler} to {batter}, FOUR!"
    if runs == 6:
        return f"{bowler} to {batter}, SIX!"
    return f"{bowler} to {batter}, {runs} run{'s' if runs > 1 else ''}."

def build_input(team, over, ball, delivery):
    extras = delivery.get("extras", {})
    wickets = delivery.get("wickets", [])

    return (
        f"Team batting: {team}\n"
        f"Over: {over}.{ball}\n"
        f"Batter: {delivery['batter']}\n"
        f"Bowler: {delivery['bowler']}\n"
        f"Runs off bat: {delivery['runs']['batter']}\n"
        f"Extras: {extras if extras else 'None'}\n"
        f"Wicket: {'Yes' if wickets else 'No'}"
    )

def process_match(file_path, out_f):
    with open(file_path, "r", encoding="utf-8") as f:
        match = json.load(f)

    for inning in match["innings"]:
        team = inning["team"]
        for over_data in inning["overs"]:
            over = over_data["over"]
            ball = 1
            for delivery in over_data["deliveries"]:
                instruction = random.choice(INSTRUCTIONS)
                input_text = build_input(team, over, ball, delivery)
                output_text = describe_delivery(delivery)

                record = {
                    "instruction": instruction,
                    "input": input_text,
                    "output": output_text
                }

                out_f.write(json.dumps(record) + "\n")
                ball += 1

def main():
    files = list(Path(DATA_DIR).glob("*.json"))
    print(f"Found {len(files)} match files")

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out_f:
        for file_path in files:
            process_match(file_path, out_f)

    print(f"Saved instruction dataset to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()


In [ ]:
# Test script to verify authentication works
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
HF_TOKEN = "YOUR_HF_TOKEN"  # Get from https://huggingface.co/settings/tokens

# First test without training
from transformers import AutoTokenizer

try:
    print("Testing tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    print("✅ Tokenizer works!")
    
    print("\nTesting model...")
    from transformers import AutoModelForCausalLM
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, token=HF_TOKEN)
    print("✅ Model works!")
    
    print("\n🎉 Authentication successful! You can now run the training script.")
    
except Exception as e:
    print(f"❌ Error: {e}")
    print("\nMake sure you:")
    print("1. Accepted terms at: https://huggingface.co/google/gemma-3-270m-it")
    print("2. Created token at: https://huggingface.co/settings/tokens")
    print("3. Copied token correctly (starts with hf_)")

In [ ]:
import torch
import gc
import os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, TaskType

# --------------------
# CONFIG
# --------------------
MODEL_NAME = "microsoft/phi-2"
DATA_PATH = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/train.jsonl"
OUTPUT_DIR = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/cricket-commentary-model"
MAX_SEQ_LEN = 512

torch.cuda.empty_cache()
gc.collect()

print(f"🚀 Training on: {torch.cuda.get_device_name(0)}")

# --------------------
# LOAD & SLICE DATASET
# --------------------
print("\n📂 Loading and slicing dataset...")
dataset = load_dataset("json", data_files=DATA_PATH, split="train")

# Using 5000 samples for high speed
dataset = dataset.shuffle(seed=42).select(range(5000)) 

def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n"
    if example.get('input'):
        prompt += f"### Input:\n{example['input']}\n\n"
    prompt += f"### Response:\n{example['output']}"
    return {"text": prompt}

dataset = dataset.map(format_example)

# --------------------
# TOKENIZER
# --------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# --------------------
# MODEL (Flash Attention Removed)
# --------------------
print("\n🤖 Loading model with SDPA...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    # 🔥 REMOVED flash_attention_2, USING sdpa INSTEAD
    attn_implementation="sdpa" 
)

peft_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "dense"], 
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# --------------------
# SFT CONFIG
# --------------------
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=True,                  
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,            
    bf16=True, 
    gradient_checkpointing=True,
    optim="adamw_torch_fused",
    logging_steps=1,
    save_steps=100,
    report_to="none",
)

# --------------------
# TRAINER
# --------------------
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer, 
)

print("\n🚀 Starting training...")
trainer.train()

# --------------------
# SAVE
# --------------------
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\n✅ Done! Model saved to {OUTPUT_DIR}")

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "microsoft/phi-2"
LORA_PATH = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/cricket-commentary-model"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# Base model
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa"
)

# Load LoRA
model = PeftModel.from_pretrained(model, LORA_PATH)
model.eval()

# Test prompt
prompt = """### Instruction:
Generate live commentary for this ball.


### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.8,
        top_p=0.9,
        do_sample=True
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


### Instruction:
Generate live commentary for this ball.


### Response:
Team batting: Chennai Super Kings
Over: 13.4
Batter: Rajasthan Royals
Bowler: Suryakumar Yadav
Runs off bat: 0
Extras: None
Wicket: No

### Commentary:
Suryakumar Yadav to Rajasthan Royals, no run.


In [ ]:
%pip install -U transformers datasets trl accelerate
%pip install sentencepiece protobuf

In [ ]:
%pip install ipywidgets
# Run this cell FIRST
from huggingface_hub import notebook_login

# This will open a widget to enter your token
notebook_login()

# OR if you want to paste token directly:
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")  # Paste your token here

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)
print(torch.__version__)
print(torch.cuda.is_available())
torch.randn(1).cuda()

x = torch.randn(10, 10, device="cuda")
y = x @ x
print(y)

In [5]:
import torch
import gc
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model, TaskType

# --------------------
# CONFIG
# --------------------
MODEL_NAME = "microsoft/phi-2"
DATA_PATH = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/gptgen.jsonl"
OUTPUT_DIR = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/cricket-fixed"
MAX_SEQ_LEN = 256 

torch.cuda.empty_cache()
gc.collect()

print(f"🚀 Training on: {torch.cuda.get_device_name(0)}")

# --------------------
# LOAD DATASET
# --------------------
print("\n📂 Loading dataset...")
dataset = load_dataset("json", data_files=DATA_PATH, split="train")
dataset = dataset.shuffle(seed=42).select(range(300))

def format_example(example):
    prompt = f"### Instruction:\n{example['instruction']}\n\n"
    if example.get('input'):
        prompt += f"### Input:\n{example['input']}\n\n"
    prompt += f"### Response:\n{example['output']}<|endoftext|>"
    return {"text": prompt}

dataset = dataset.map(format_example)

# --------------------
# TOKENIZER
# --------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

# --------------------
# MODEL - FIX FOR "META DEVICE" ERROR
# --------------------
print("\n🤖 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"": 0}, # FORCE everything onto GPU 0 to avoid meta-device splits
    trust_remote_code=True,
    # Removed sdpa as it often causes the meta-tensor conflict on Phi-2
)

# --------------------
# LoRA CONFIG
# --------------------
peft_config = LoraConfig(
    r=16, 
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "dense", "fc1", "fc2"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, peft_config)

# --------------------
# SFT CONFIG - STABILIZED
# --------------------
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    packing=False,
    per_device_train_batch_size=2,      # Reduced to 2
    gradient_accumulation_steps=8,     # Increased to 8 (Total Batch = 16)
    learning_rate=5e-5,                # Cut in half for stability
    lr_scheduler_type="cosine",        # Gradually lowers LR to help convergence
    warmup_ratio=0.1,                  # Slowly ramps up at the start
    num_train_epochs=5, 
    fp16=True,
    gradient_checkpointing=False,
    optim="adamw_torch",
    logging_steps=1,
    save_steps=50,
    report_to="none",
)

# --------------------
# TRAINER
# --------------------
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print(f"\n🚀 Starting training with Meta-Device fix...")
trainer.train()

# --------------------
# SAVE
# --------------------
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n✅ Done!")

🚀 Training on: NVIDIA GeForce RTX 5080 Laptop GPU

📂 Loading dataset...

🤖 Loading model...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.



🚀 Starting training with Meta-Device fix...


Step,Training Loss
1,3.472100
2,3.611700
3,3.535300
4,3.609400
5,3.517400
6,3.522800
7,3.575500
8,3.471000
9,3.432100
10,3.281900



✅ Done!


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Same paths as your training script
MODEL_NAME = "microsoft/phi-2"
ADAPTER_PATH = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/cricket-fixed"

print("loading model and adapter...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# Load Base Model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map={"": 0},
    trust_remote_code=True
)

# Load your Fine-tuned Adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

def ask_cricket(instruction, input_text=""):
    # Match the exact format used in training
    prompt = f"### Instruction:\n{instruction}\n\n"
    if input_text:
        prompt += f"### Input:\n{input_text}\n\n"
    prompt += "### Response:\n"
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            eos_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Clean up the output to show only the response
    return response.split("### Response:\n")[-1].strip()

# --- TEST IT ---
question = "describe a tense first over" # Change this to a question in your jsonl
print(f"\n🏏 Query: {question}")
print(f"🤖 Answer: {ask_cricket(question)}")

loading model and adapter...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



🏏 Query: describe a tense first over
🤖 Answer: The tension is palpable as the first over begins. Both teams are focused and alert, waiting for the first ball to be bowled. The bowler is nervous, knowing that the first ball will set the tone for the entire over. The batter is equally anxious, trying to remain calm and focused.


In [ ]:
import torch
import json
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from difflib import SequenceMatcher

# --------------------
# PATHS
# --------------------
BASE_MODEL = "microsoft/phi-2"
LORA_PATH = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/cricket"
DATA_PATH = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/gptgen.jsonl"

# --------------------
# LOAD DATASET
# --------------------
examples = []
with open(DATA_PATH, 'r') as f:
    for line in f:
        examples.append(json.loads(line))

# --------------------
# TOKENIZER
# --------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# --------------------
# MODEL + LoRA
# --------------------
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="sdpa"
)
model = PeftModel.from_pretrained(model, LORA_PATH)
model.eval()

# --------------------
# HELPER: similarity score
# --------------------
def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio() * 100

# --------------------
# TEST ALL EXAMPLES
# --------------------
scores = []
for i, ex in enumerate(examples, 1):
    prompt = f"### Instruction:\n{ex['instruction']}\n\n"
    if ex.get("input"):
        prompt += f"### Input:\n{ex['input']}\n\n"
    prompt += "### Response:\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.8,
            top_p=0.9,
            do_sample=True
        )

    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    # Only take the generated part after "### Response:"
    gen_response = generated_text.split("### Response:")[-1].strip()

    sim_score = similarity(ex['output'], gen_response)
    scores.append(sim_score)

    print(f"\n=== Example {i} ===")
    print(f"Instruction: {ex['instruction']}")
    if ex.get("input"):
        print(f"Input: {ex['input']}")
    print(f"\n--- Original Output ---\n{ex['output']}")
    print(f"\n--- Model Generated ---\n{gen_response}")
    print(f"\nSimilarity: {sim_score:.2f}%")

# --------------------
# SUMMARY
# --------------------
avg_score = sum(scores) / len(scores)
low_examples = sum(1 for s in scores if s < 50)

print("\n" + "="*40)
print(f"✅ Average similarity: {avg_score:.2f}%")
print(f"Examples with <50% similarity: {low_examples} / {len(examples)}")
print("="*40)


In [12]:
import json
import os

# -----------------------------
# CONFIG
# -----------------------------
MATCH_JSON_FOLDER = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/data"  # Folder with JSON match files
OUTPUT_JSONL = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/cricket_match_summary.jsonl"

# -----------------------------
# Helper function: JSON -> single fact chunk per match
# -----------------------------
def json_to_single_fact(match_json, match_id=None):
    """
    Converts a single match JSON to ONE fact chunk summarizing:
    - Teams
    - Winner and margin
    - Number of overs
    - Player(s) of the match
    - Toss
    - Venue
    - Date
    """
    info = match_json.get("info", {})
    event = info.get("event", {})
    outcome = info.get("outcome", {})
    
    # Season
    season = None
    if info.get("season"):
        try:
            season = int(str(info["season"]).split("/")[0])
        except:
            season = None
    if not season and info.get("dates"):
        season = int(info["dates"][0][:4])
    if not season:
        season = 0
    
    # Teams
    teams = info.get("teams", [])
    team_text = ""
    if teams and len(teams) == 2:
        team_text = f"The match was between {teams[0]} and {teams[1]}."
    
    # Winner + margin
    winner_text = ""
    if outcome:
        winner = outcome.get("winner")
        by = outcome.get("by", {})
        margin_parts = []
        if "wickets" in by:
            margin_parts.append(f"{by['wickets']} wickets")
        if "runs" in by:
            margin_parts.append(f"{by['runs']} runs")
        if winner and margin_parts:
            winner_text = f"{winner} won by {' and '.join(margin_parts)}."
    
    # Overs
    overs_text = ""
    if info.get("overs"):
        overs_text = f"The match had {info['overs']} overs."
    
    # Player of the match
    pom_text = ""
    if info.get("player_of_match"):
        pom_text = "Player of the match was " + ", ".join(info["player_of_match"]) + "."
    
    # Toss
    toss_text = ""
    toss = info.get("toss", {})
    if toss.get("winner") and toss.get("decision"):
        toss_text = f"{toss['winner']} won the toss and decided to {toss['decision']}."
    
    # Venue
    venue_text = ""
    if info.get("venue"):
        venue_text = f"The match was played at {info['venue']}."
    
    # Date
    date_text = ""
    if info.get("dates"):
        date_text = f"The match took place on {', '.join(info['dates'])}."
    
    # Combine everything
    full_text = " ".join([team_text, winner_text, overs_text, pom_text, toss_text, venue_text, date_text]).strip()
    
    return {
        "text": full_text,
        "match_id": match_id,
        "season": season
    }

# -----------------------------
# Process all JSON files
# -----------------------------
all_facts = []

for filename in os.listdir(MATCH_JSON_FOLDER):
    if filename.endswith(".json"):
        filepath = os.path.join(MATCH_JSON_FOLDER, filename)
        with open(filepath, "r") as f:
            match_json = json.load(f)
        
        match_id = filename.replace(".json", "")
        fact = json_to_single_fact(match_json, match_id=match_id)
        all_facts.append(fact)

# -----------------------------
# Save to JSONL
# -----------------------------
with open(OUTPUT_JSONL, "w") as f:
    for fact in all_facts:
        f.write(json.dumps(fact) + "\n")

print(f"✅ Done! Total matches processed: {len(all_facts)}")
print(f"Saved to: {OUTPUT_JSONL}")


✅ Done! Total matches processed: 1169
Saved to: /mnt/c/Users/pragn/Downloads/Gemma_finetune/cricket_match_summary.jsonl


In [14]:
%pip install --no-deps threadpoolctl



import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

import torch

# -----------------------------
# CONFIG
# -----------------------------
FACTS_FILE = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/cricket_match_summary.jsonl"
LORA_PATH = "/mnt/c/Users/pragn/Downloads/Gemma_finetune/cricket"
BASE_MODEL = "microsoft/phi-2"
TOP_K = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 200

# -----------------------------
# Load fact chunks
# -----------------------------
facts = []
metadata = []

with open(FACTS_FILE, "r") as f:
    for line in f:
        obj = json.loads(line)
        facts.append(obj["text"])
        metadata.append(obj)  # store match_id, season, city etc.

print(f"✅ Loaded {len(facts)} facts.")

# -----------------------------
# Embed facts using SentenceTransformer
# -----------------------------
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = embedder.encode(facts, convert_to_numpy=True, normalize_embeddings=True)

# Build FAISS index
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)  # Inner-product for cosine similarity
index.add(embeddings)
print(f"✅ FAISS index built with {index.ntotal} vectors.")

# -----------------------------
# Load LoRA Phi-2 model
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

# Load base model
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# Load LoRA weights
model = PeftModel.from_pretrained(model, LORA_PATH)
model.to(DEVICE)
model.eval()
print(f"✅ LoRA Phi-2 model loaded from {LORA_PATH}.")

# -----------------------------
# Helper: Retrieve top-k facts
# -----------------------------
def retrieve_facts(query, top_k=TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    D, I = index.search(q_emb, top_k)
    return [metadata[i]["text"] for i in I[0]]

# -----------------------------
# Helper: Build prompt for RAG
# -----------------------------
def build_rag_prompt(query, retrieved_facts):
    context_text = "\n".join(f"- {f}" for f in retrieved_facts)
    prompt = f"""
### Instruction:
Answer the user query using only the facts provided below. Do not hallucinate.

### Context:
{context_text}

### Input:
{query}

### Response:
"""
    return prompt

# -----------------------------
# Example: Query RAG
# -----------------------------
user_query = "Who won the IPL 2016 Qualifier 1 and who was player of the match?"
retrieved = retrieve_facts(user_query)
prompt = build_rag_prompt(user_query, retrieved)

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n✅ Retrieved Facts:\n")
for f in retrieved:
    print("-", f)

print("\n✅ RAG Answer:\n")
print(answer)

# -----------------------------
# Optional: Interactive loop
# -----------------------------
while True:
    query = input("\nAsk a cricket question (or 'exit'): ")
    if query.lower() == "exit":
        break
    retrieved = retrieve_facts(query)
    prompt = build_rag_prompt(query, retrieved)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print("\nAnswer:\n", answer)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.
✅ Loaded 1169 facts.
✅ FAISS index built with 1169 vectors.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


✅ LoRA Phi-2 model loaded from /mnt/c/Users/pragn/Downloads/Gemma_finetune/cricket.

✅ Retrieved Facts:

- The match was between Sunrisers Hyderabad and Kolkata Knight Riders. Sunrisers Hyderabad won by 22 runs. The match had 20 overs. Player of the match was MC Henriques. Kolkata Knight Riders won the toss and decided to field. The match was played at Feroz Shah Kotla. The match took place on 2016-05-25.
- The match was between Sunrisers Hyderabad and Kings XI Punjab. Sunrisers Hyderabad won by 5 wickets. The match had 20 overs. Player of the match was Mustafizur Rahman. Sunrisers Hyderabad won the toss and decided to field. The match was played at Rajiv Gandhi International Stadium, Uppal. The match took place on 2016-04-23.
- The match was between Gujarat Lions and Sunrisers Hyderabad. Sunrisers Hyderabad won by 8 wickets. The match had 20 overs. Player of the match was Mohammed Siraj. Sunrisers Hyderabad won the toss and decided to field. The match was played at Green Park. The mat

...


Answer:
 
### Instruction:
Answer the user query using only the facts provided below. Do not hallucinate.

### Context:
- The match was between Royal Challengers Bangalore and Kings XI Punjab. Royal Challengers Bangalore won by 138 runs. The match had 20 overs. Player of the match was CH Gayle. Kings XI Punjab won the toss and decided to field. The match was played at M Chinnaswamy Stadium. The match took place on 2015-05-06.
- The match was between Royal Challengers Bangalore and Delhi Daredevils.  The match had 20 overs. Player of the match was V Kohli. Royal Challengers Bangalore won the toss and decided to field. The match was played at M Chinnaswamy Stadium. The match took place on 2013-04-16.
- The match was between Royal Challengers Bangalore and Kings XI Punjab. Royal Challengers Bangalore won by 85 runs. The match had 20 overs. Player of the match was CH Gayle. Kings XI Punjab won the toss and decided to field. The match was played at M Chinnaswamy Stadium. The match took pla

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Answer:
 
### Instruction:
Answer the user query using only the facts provided below. Do not hallucinate.

### Context:
- The match was between Gujarat Lions and Kings XI Punjab. Kings XI Punjab won by 26 runs. The match had 20 overs. Player of the match was HM Amla. Gujarat Lions won the toss and decided to field. The match was played at Saurashtra Cricket Association Stadium. The match took place on 2017-04-23.
- The match was between Gujarat Lions and Royal Challengers Bangalore. Royal Challengers Bangalore won by 21 runs. The match had 20 overs. Player of the match was CH Gayle. Gujarat Lions won the toss and decided to field. The match was played at Saurashtra Cricket Association Stadium. The match took place on 2017-04-18.
- The match was between Gujarat Lions and Royal Challengers Bangalore. Gujarat Lions won by 6 wickets. The match had 20 overs. Player of the match was V Kohli. Royal Challengers Bangalore won the toss and decided to bat. The match was played at Saurashtra Cric

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



Answer:
 
### Instruction:
Answer the user query using only the facts provided below. Do not hallucinate.

### Context:
- The match was between Gujarat Lions and Kings XI Punjab. Kings XI Punjab won by 26 runs. The match had 20 overs. Player of the match was HM Amla. Gujarat Lions won the toss and decided to field. The match was played at Saurashtra Cricket Association Stadium. The match took place on 2017-04-23.
- The match was between Gujarat Lions and Royal Challengers Bangalore. Royal Challengers Bangalore won by 21 runs. The match had 20 overs. Player of the match was CH Gayle. Gujarat Lions won the toss and decided to field. The match was played at Saurashtra Cricket Association Stadium. The match took place on 2017-04-18.
- The match was between Gujarat Lions and Royal Challengers Bangalore. Gujarat Lions won by 6 wickets. The match had 20 overs. Player of the match was V Kohli. Royal Challengers Bangalore won the toss and decided to bat. The match was played at Saurashtra Cric